## FACT FOLLOWS

In [0]:
import dlt
import pyspark.sql.functions as F

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7790558950369495>, line 1
----> 1 import dlt
      2 import pyspark.sql.functions as F

ModuleNotFoundError: No module named 'dlt'

#### Data Reading

In [0]:
import dlt
import pyspark.sql.functions as F


@dlt.table(
    name="fact_user_follow",
    comment="Factless fact table recording user follow relationships without surrogate keys",
    table_properties={"quality": "gold"},
)
def fact_user_follow():
    follows_df = dlt.read("travel_journal_catalog.silver.follows")
    dim_users = dlt.read("dim_user")

    follows_prepared = (
        follows_df.withColumn("created_date", F.to_date(F.col("created_at")))
        .withColumnRenamed("flag", "is_flagged")
        .withColumn("follow_count", F.lit(1))
        .alias("follows_df")
    )

    # 1. Join for Follower (using your 'follwer_dim' alias)
    follower_joined = follows_prepared.join(
        dim_users.alias("follwer_dim"),
        (F.col("follows_df.follower_id") == F.col("follwer_dim.user_id"))
        & (F.col("follows_df.created_at") >= F.col("follwer_dim.__START_AT"))
        & (
            (F.col("follows_df.created_at") < F.col("follwer_dim.__END_AT"))
            | F.col("follwer_dim.__END_AT").isNull()
        ),
        how="inner",
    ).select(
        F.col("follows_df.id").alias("id"),
        F.col("follwer_dim.DimUserKey").alias("follower_key"),
        F.col("follwer_dim.user_id").alias("follower_user_id"),
        F.col("follwer_dim.__START_AT").alias("follower_user_version_at"),
        F.col("follows_df.following_id"),
        F.col("follows_df.created_at"),
        F.col("follows_df.is_flagged"),
        F.col("follows_df.follow_count")
    )

# 3. Join for Following (using explicit alias 'fj' to match follower_joined)
    fact_df = (
        follower_joined.alias("fj").join(
            dim_users.alias("following_dim"),
            (F.col("fj.following_id") == F.col("following_dim.user_id"))
            & (F.col("fj.created_at") >= F.col("following_dim.__START_AT"))
            & (
                (F.col("fj.created_at") < F.col("following_dim.__END_AT"))
                | F.col("following_dim.__END_AT").isNull()
            ),
            how="inner",
        )
        .select(
            F.col("fj.id"),
            F.col("fj.follower_key"),
            F.col("fj.follower_user_id"),
            F.col("fj.follower_user_version_at"),
            F.col("following_dim.user_id").alias("following_user_id"),
            F.col("following_dim.DimUserKey").alias("following_key"),
            F.date_format(F.col("fj.created_at"), "yyyyMMdd").cast("int").alias("follow_date_id"),
            F.col("fj.created_at"),
            F.col("fj.is_flagged"),
            F.col("fj.follow_count"),
        )
    )


    return fact_df

###